# Pandas Part 1: DataFrames, Reading CSV, Selection & Indexing

Pandas is the primary tool for working with tabular data in Python. Its core data structure, the **DataFrame**, is a two-dimensional table of rows and columns — think of it as a spreadsheet you can manipulate programmatically. A **Series** is a single column (or row) from a DataFrame. In Tabular ML workflows, pandas handles almost everything before modeling: loading data, exploring it, and engineering features. The operations in this notebook are the foundation you will use in most of the tabular projects.

<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>What you will learn</b>

- Load a CSV file into a DataFrame
- Inspect a new dataset: shape, types, summary statistics, missing values
- Select individual columns and multiple columns
- Select rows by label (<code>.loc</code>) and by position (<code>.iloc</code>)

<b>Dataset:</b> AI/ML Job Salaries — global salary data for data science roles (2020-2024).

---

<b>How to use this notebook</b>

1. Run cells top to bottom with <code>Shift + Enter</code>.
2. <b>TODO cells</b> are exercises — try to solve before expanding the solution.
3. Solutions are hidden in collapsible blocks.

</div>

---
## Step 1: Setup

In [1]:
import pandas as pd

print("Pandas:", pd.__version__)

Pandas: 3.0.3


---
## Step 2: Loading and saving Data

`pd.read_csv()` is the most common way to load data. It reads a CSV file and returns a DataFrame.

In [2]:
df = pd.read_csv('salaries.csv')
df.head()  # Display the first few rows of the DataFrame

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M
1,2025,SE,FT,Data Product Owner,110000,USD,110000,US,0,US,M
2,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M
3,2025,SE,FT,Data Product Owner,110000,USD,110000,US,0,US,M
4,2025,SE,FT,Engineer,143000,USD,143000,US,0,US,M


### Dataset columns

| Column | Description |
|---|---|
| `work_year` | Year the salary was paid |
| `experience_level` | EN = Junior, MI = Mid, SE = Senior, EX = Executive |
| `employment_type` | FT = Full-time, PT = Part-time, CT = Contract, FL = Freelance |
| `job_title` | Role title |
| `salary` | Gross salary in the original currency |
| `salary_currency` | Currency code |
| `salary_in_usd` | Salary converted to USD |
| `employee_residence` | Country of employee residence (ISO code) |
| `remote_ratio` | 0 = on-site, 50 = hybrid, 100 = fully remote |
| `company_location` | Country of company headquarters |
| `company_size` | S = small (<50), M = medium (50-250), L = large (>250) |

### read_csv options worth knowing

In [3]:
# Load only specific columns (saves memory on large files)
df_subset = pd.read_csv('salaries.csv', usecols=['work_year', 'job_title', 'salary_in_usd', 'experience_level'])
print("Columns loaded:", df_subset.columns.tolist())
print("Shape:", df_subset.shape)

Columns loaded: ['work_year', 'experience_level', 'job_title', 'salary_in_usd']
Shape: (73148, 4)


> **Note:** For large datasets, `usecols` reduces memory usage significantly — you only load what you need. Pandas can also read other formats with the same pattern: `pd.read_excel()`, `pd.read_json()`, `pd.read_parquet()`, `pd.read_sql()`.

### Save a DataFrame to CSV

In [4]:
df_subset.to_csv('output.csv', index=False)  # index=False avoids writing row numbers as an extra column
print("Saved to output.csv")

Saved to output.csv


> **Note:** `to_csv('filename.csv', index=False)` is the standard way to save. `index=False` prevents pandas from writing the row numbers (0, 1, 2…) as an extra column in the file — you almost always want this flag.

### Inspect the dataset

For the rest of this tutorial, we use the full dataset:

In [5]:
df = pd.read_csv('salaries.csv')

---
## Step 3: Exploring Your DataFrame

Every time you load a new dataset, run these commands first. They answer the four questions you always need to start with: *How big is it? What types are the columns? Are there missing values? What do the summary statistics look like?*

In [16]:
# 1. How many rows and columns?
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (73148, 11)


In [17]:
# 2. Column names and data types
print(df.dtypes)

work_year             int64
experience_level        str
employment_type         str
job_title               str
salary                int64
salary_currency         str
salary_in_usd         int64
employee_residence      str
remote_ratio          int64
company_location        str
company_size            str
dtype: object


> **Note:** In pandas 3.x, text/string columns have dtype `str`. Numeric columns are `int64` (whole numbers) or `float64` (decimals).

In [18]:
# 3. Compact summary: types + non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73148 entries, 0 to 73147
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   work_year           73148 non-null  int64
 1   experience_level    73148 non-null  str  
 2   employment_type     73148 non-null  str  
 3   job_title           73148 non-null  str  
 4   salary              73148 non-null  int64
 5   salary_currency     73148 non-null  str  
 6   salary_in_usd       73148 non-null  int64
 7   employee_residence  73148 non-null  str  
 8   remote_ratio        73148 non-null  int64
 9   company_location    73148 non-null  str  
 10  company_size        73148 non-null  str  
dtypes: int64(4), str(7)
memory usage: 6.1 MB


In [19]:
# 4. Summary statistics for numerical columns
df.describe()

,work_year,salary,salary_in_usd,remote_ratio
count,73148.000000,7.314800e+04,73148.000000,73148.000000
mean,2023.831192,1.625534e+05,158013.748619,21.582955
std,0.477551,1.925761e+05,72501.304728,41.023051
min,2020.000000,1.400000e+04,15000.000000,0.000000
25%,2024.000000,1.069575e+05,106890.000000,0.000000
50%,2024.000000,1.480000e+05,147500.000000,0.000000
75%,2024.000000,2.000000e+05,199700.000000,0.000000
max,2025.000000,3.040000e+07,800000.000000,100.000000


In [20]:
# Summary statistics for categorical columns
df.describe(include='str')

,experience_level,employment_type,job_title,salary_currency,employee_residence,company_location,company_size
count,73148,73148,73148,73148,73148,73148,73148
unique,4,4,289,25,93,86,3
top,SE,FT,Data Scientist,USD,US,US,M
freq,42926,72808,11443,69418,65982,66035,70536


> **Note:** `NaN` (Not a Number) is pandas' representation of a missing value — a cell that contains no data. You will see it wherever data is absent.

In [21]:
# 5. Missing values
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64


In [22]:
# First and last rows
print("First 3 rows:")
display(df.head(3))
print("\nLast 3 rows:")
display(df.tail(3))

First 3 rows:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M
1,2025,SE,FT,Data Product Owner,110000,USD,110000,US,0,US,M
2,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M



Last 3 rows:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
73145,2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S
73146,2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L
73147,2021,SE,FT,Data Scientist,7000000,INR,94665,IN,50,IN,L


In [23]:
# Random sample — useful for spotting variety in large datasets without always seeing the same top rows
df.sample(5, random_state=42)

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
484,2024,MI,FT,Data Management Specialist,123000,USD,123000,US,0,US,M
68397,2023,SE,FT,Data Analyst,186600,USD,186600,US,100,US,M
20934,2024,EN,FT,Data Analyst,91800,USD,91800,US,0,US,M
1583,2024,SE,FT,Data Lead,114720,USD,114720,US,0,US,M
15939,2024,MI,FT,Data Manager,64600,USD,64600,US,0,US,M


**TODO:** How many unique job titles are in the dataset? Use `.nunique()` to find out.

In [24]:
# YOUR TURN
print("Unique job titles:", df['job_title'].nunique())

Unique job titles: 289


<details><summary><b>Solution — click to expand</b></summary>

```python
print("Unique job titles:", df['job_title'].nunique())

# Bonus
print("\nTop 10 most common:")
print(df['job_title'].value_counts().head(10))
```
</details>

---
## Step 4: Column Selection

Two ways to select columns — single column returns a **Series**, multiple columns returns a **DataFrame**.

> **The DataFrame index:** Every DataFrame has an index — a row label visible on the left side of output. By default it's `0, 1, 2, ...` (a `RangeIndex`). After operations like `groupby().reset_index()`, it resets to sequential integers. You can safely ignore the index for now; just know it exists and that `.loc` uses it for row selection.

In [25]:
# Single column -> Series
salary_series = df['salary_in_usd']
print(type(salary_series))
print(salary_series.head())

<class 'pandas.Series'>
0    170000
1    110000
2    170000
3    110000
4    143000
Name: salary_in_usd, dtype: int64


In [26]:
# Multiple columns -> DataFrame (note the double brackets)
df_selected = df[['job_title', 'experience_level', 'salary_in_usd']]
print(type(df_selected))
df_selected.head()

<class 'pandas.DataFrame'>


,job_title,experience_level,salary_in_usd
0,Data Product Owner,SE,170000
1,Data Product Owner,SE,110000
2,Data Product Owner,SE,170000
3,Data Product Owner,SE,110000
4,Engineer,SE,143000


**TODO:** Select only the `work_year`, `company_size`, and `salary_in_usd` columns and display the first 5 rows.

In [27]:
# YOUR TURN
df_selected = df[['work_year','company_size','salary_in_usd']]
print(type(df_selected))
df_selected.head()

<class 'pandas.DataFrame'>


,work_year,company_size,salary_in_usd
0,2025,M,170000
1,2025,M,110000
2,2025,M,170000
3,2025,M,110000
4,2025,M,143000


<details><summary><b>Solution — click to expand</b></summary>

```python
df[['work_year', 'company_size', 'salary_in_usd']].head()
```
</details>

---
## Step 5: Row Selection with .loc and .iloc

Two indexers for selecting rows (and optionally columns):

- **`.loc`** — label-based: uses column names and index labels
- **`.iloc`** — position-based: uses integer positions (like Python list slicing)

The key difference: `.loc` is inclusive of both endpoints; `.iloc` excludes the stop, like Python slices.

In [28]:
# .loc — select by label
# Select rows with index labels from 0 to 2 (inclusive) and specific columns. 
# Reminder: indexing starts at 0, so this will give us the first three rows.
df.loc[0:2, ['job_title', 'experience_level', 'salary_in_usd']]

,job_title,experience_level,salary_in_usd
0,Data Product Owner,SE,170000
1,Data Product Owner,SE,110000
2,Data Product Owner,SE,170000


In [29]:
# .iloc — select by integer position
# Select first 2 rows (exclusive, positions 0 and 1) and first 4 columns (positions 0-3)
df.iloc[0:2, 0:4]

,work_year,experience_level,employment_type,job_title
0,2025,SE,FT,Data Product Owner
1,2025,SE,FT,Data Product Owner


In [ ]:
# .iloc with specific positions (non-consecutive)
# Select rows 0, 5, 10 and columns 0, 3 (work_year and job_title)
df.iloc[[0, 5, 10], [0, 3]]

,work_year,job_title
0,2025,Data Product Owner
5,2025,Engineer
10,2025,Data Scientist


In [ ]:
# Select a single value
print(".loc single value:", df.loc[0, 'job_title'])
print(".iloc single value:", df.iloc[0, 3])  # row 0, column position 3

.loc single value: Data Product Owner
.iloc single value: Data Product Owner


<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>Best practice: prefer .loc over chained indexing for assignments</b>

Always use <code>df.loc[mask, 'column'] = value</code> for assignments, not <code>df[mask]['column'] = value</code> (which is also possible in pandas).

Chained indexing may modify a copy rather than the original DataFrame, producing a <code>SettingWithCopyWarning</code> or silently failing. <code>.loc</code> always modifies the original in place.

</div>

**TODO:** Use `.iloc` to select the last 3 rows of the DataFrame and the first 3 columns.

In [33]:
# YOUR TURN
# Hint: use negative indexing for the last N rows
df.iloc[-3:, :3]

,work_year,experience_level,employment_type
73145,2020,EN,FT
73146,2020,EN,CT
73147,2021,SE,FT


<details><summary><b>Solution — click to expand</b></summary>

```python
df.iloc[-3:, :3]
```
</details>

**TODO:** Use `.loc` to select rows 10 through 14 and display only the `job_title`, `experience_level`, and `salary_in_usd` columns.

In [34]:
# YOUR TURN
df.loc[10:14, ['job_title', 'experience_level', 'salary_in_usd']]

,job_title,experience_level,salary_in_usd
10,Data Scientist,SE,234000
11,Data Scientist,SE,146000
12,Data Scientist,SE,86000
13,Data Scientist,SE,80000
14,Data Scientist,EN,126000


<details><summary><b>Solution — click to expand</b></summary>

```python
df.loc[10:14, ['job_title', 'experience_level', 'salary_in_usd']]
```
</details>

---
## Summary

You can now:
- Load a CSV file with `pd.read_csv()` and save with `df.to_csv('file.csv', index=False)`
- Inspect a DataFrame: `shape`, `dtypes`, `info()`, `describe()`, `isnull().sum()`
- Select columns: `df['col']` (Series) and `df[['col1', 'col2']]` (DataFrame)
- Select rows by label: `df.loc[rows, cols]`
- Select rows by position: `df.iloc[rows, cols]`